# BrowniGAT Demo Analysis

This notebook demonstrates the current public-facing BrowniGAT stack in three connected steps:

1. Run the `vNext` multimodal pipeline
2. Materialize the foundation scheduler control plane
3. Generate synthetic biology design outputs

The notebook is designed to be reproducible with the repository toy data and example configs.

In [ ]:
from pathlib import Path
import json
import subprocess

import pandas as pd

REPO_ROOT = Path.cwd()
DEMO_ROOT = REPO_ROOT / "results" / "notebook_demo"
VNEXT_DIR = DEMO_ROOT / "vnext"
SCHED_DIR = DEMO_ROOT / "scheduler"
SYNBIO_DIR = DEMO_ROOT / "synbio"
DEMO_ROOT.mkdir(parents=True, exist_ok=True)
DEMO_ROOT

## 1. Run the vNext multimodal pipeline

In [ ]:
subprocess.run(
    ["python", "vnext_main.py", "--config", "config/vnext_toy.yaml", "--output-dir", str(VNEXT_DIR)],
    check=True,
)
causal_df = pd.read_csv(VNEXT_DIR / "causal_ranking.tsv", sep="\t")
causal_df.head(5)

## 2. Build the canonical bundle and run the scheduler control plane

In [ ]:
subprocess.run(
    ["python", "ingest_multimodal_data.py", "--config", "config/real_data_example.yaml"],
    check=True,
)
subprocess.run(
    ["python", "foundation_schedule.py", "--config", "config/foundation_engine_example.yaml", "--workspace-dir", str(SCHED_DIR)],
    check=True,
)
scheduler_summary = json.loads((SCHED_DIR / "scheduler" / "scheduler_summary.json").read_text(encoding="utf-8"))
scheduler_summary

In [ ]:
run_queue = json.loads((SCHED_DIR / "scheduler" / "run_queue.json").read_text(encoding="utf-8"))
pd.DataFrame(run_queue)[["queue_id", "stage_name", "priority", "stage_kind", "status", "resource_request"]]

## 3. Generate synthetic biology design outputs

In [ ]:
subprocess.run(
    ["python", "synbio_main.py", "--config", "config/synbio_toy.yaml", "--output-dir", str(SYNBIO_DIR)],
    check=True,
)
construct_df = pd.read_csv(SYNBIO_DIR / "construct_blueprints.tsv", sep="\t")
construct_df.head(5)

## 4. Suggested interpretation

- `causal_ranking.tsv` shows the highest-confidence intervention-oriented targets.
- `scheduler_summary.json` shows the control-plane view of stage execution.
- `construct_blueprints.tsv` demonstrates how BrowniGAT can move from ranking into design-oriented synthetic biology outputs.